In [236]:
# Loading the data
import json
import pandas as pd
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from sklearn.covariance import MinCovDet
import numpy as np

# Define the project root (one folder up from /workshop)
project_root = Path("..").resolve()

# Add the project root to sys.path so Jupyter can find your 'src' module
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Load data/valencia_cf_1/matches.json
matches_path = project_root / "tests" / "fixtures" / "testing_data" / "data" / "valencia_cf_1" / "matches.json"
raw_matches = json.load(open(matches_path))
print(f"Loaded {len(raw_matches)} matches from {matches_path}")

Loaded 100 matches from C:\Users\kazik\Projects\Gaffers-Clipboard\tests\fixtures\testing_data\data\valencia_cf_1\matches.json


In [237]:
normalized_df = pd.json_normalize(
    raw_matches,
    record_path="player_performances",
    meta=[
        "id", ["data", "half_length"], 
        ["data", "home_team_name"], 
        ["data", "away_team_name"], 
        ["data", "home_stats", "possession"], 
        ["data", "away_stats", "possession"],
        ["data", "home_stats", "xg"],
        ["data", "away_stats", "xg"],
    ]
)

normalized_df["team_possession"] = float("nan")
normalized_df["team_xg"] = float("nan")
normalized_df["opponent_xg"] = float("nan")

for idx, row in normalized_df.iterrows():
    if row["data.home_team_name"] == "Valencia CF":
        normalized_df.at[idx, "team_possession"] = row["data.home_stats.possession"]
        normalized_df.at[idx, "team_xg"] = row["data.home_stats.xg"]
        normalized_df.at[idx, "opponent_xg"] = row["data.away_stats.xg"]
    else:
        normalized_df.at[idx, "team_possession"] = row["data.away_stats.possession"]
        normalized_df.at[idx, "team_xg"] = row["data.away_stats.xg"]
        normalized_df.at[idx, "opponent_xg"] = row["data.home_stats.xg"]

normalized_df = normalized_df.drop(columns=[
    "data.home_team_name", "data.away_team_name", "data.home_stats.possession", "data.away_stats.possession", "data.home_stats.xg", "data.away_stats.xg"])

normalized_df = normalized_df.rename(columns={
    "data.half_length": "half_length",
    "id": "match_id",
})

normalized_df = normalized_df[normalized_df["performance_type"] != "GK"]

normalized_df = normalized_df.dropna(axis=1, how="all")

st_df = normalized_df[normalized_df["positions_played"].apply(lambda x: x == ["ST"])]

st_df = st_df.dropna(axis=1, how="all")

st_df = st_df.drop(columns=["performance_type", "positions_played"])

cols = st_df.columns.tolist()
cols.insert(0, cols.pop(cols.index("match_id")))
st_df = st_df[cols]

print(f"Number of rows: {st_df.shape[0]}")
print(f"Number of columns: {st_df.shape[1]}")
print("Columns:")
for col in cols:
    print(f" - {col}")

# Output the max and min of every column, to get a sense of the range of values
for col in st_df.columns:
    if col in ["match_id", "player_id"]:
        continue
    if st_df[col].dtype in ["int64", "float64"]:
        print(f"{col}: min={st_df[col].min()}, max={st_df[col].max()}")

Number of rows: 151
Number of columns: 23
Columns:
 - match_id
 - player_id
 - goals
 - assists
 - shots
 - shot_accuracy
 - passes
 - pass_accuracy
 - dribbles
 - dribble_success_rate
 - tackles
 - tackle_success_rate
 - offsides
 - fouls_committed
 - possession_won
 - possession_lost
 - minutes_played
 - distance_covered
 - distance_sprinted
 - half_length
 - team_possession
 - team_xg
 - opponent_xg
goals: min=0.0, max=5.0
assists: min=0.0, max=2.0
shots: min=0.0, max=10.0
shot_accuracy: min=0.0, max=100.0
passes: min=0.0, max=31.0
pass_accuracy: min=0.0, max=100.0
dribbles: min=0.0, max=30.0
dribble_success_rate: min=0.0, max=100.0
tackles: min=0.0, max=6.0
tackle_success_rate: min=0.0, max=100.0
offsides: min=0.0, max=2.0
fouls_committed: min=0.0, max=2.0
possession_won: min=0.0, max=4.0
possession_lost: min=0.0, max=12.0
minutes_played: min=4.0, max=93.0
distance_covered: min=0.4, max=13.3
distance_sprinted: min=0.1, max=5.4
team_possession: min=39.0, max=71.0
team_xg: min=0.7, m

In [238]:
# 1. Select every column EXCEPT your identifiers
cols_to_convert = st_df.columns.drop(['match_id', 'player_id'])

# 2. Force them all to numeric in one swoop
st_df[cols_to_convert] = st_df[cols_to_convert].apply(pd.to_numeric, errors='coerce')

# Optional: If any weird JSON text like "N/A" got turned into a NaN, 
# you can safely fill them with 0s to keep the math from breaking later.
st_df[cols_to_convert] = st_df[cols_to_convert].fillna(0)

In [239]:
# Step 1: Removing players with less than 10 minutes played
st_df = st_df[st_df["minutes_played"] >= 10]

In [240]:
# Step 1: Volume masking
# For each percentage stat, their associated volume attempted stat must reach a certain threshold for the percentage 
# stat to be considered valid. For example, if a player attempted 0 passes, their pass accuracy should not be considered valid,
# and should be set to the mean.
volume_perc_pairs = [
    ("passes", "pass_accuracy"),
    ("shots", "shot_accuracy"),
    ("dribbles", "dribble_success_rate"),
    ("tackles", "tackle_success_rate")
]
volume_masks = {
    "passes": 5,
    "shots": 2,
    "dribbles": 3,
    "tackles": 2
}

for volume_col, perc_col in volume_perc_pairs:
    # apply the mask
    valid = st_df[st_df[volume_col] >= volume_masks[volume_col]]
    mean_value = valid[perc_col].mean()
    st_df[perc_col] = st_df.apply(
        lambda r: r[perc_col] if r[volume_col] >= volume_masks[volume_col] else mean_value,
        axis=1
    )

In [241]:
# ---------------------------------------------------------
# Step 1: Standardize Absolute Volume (The Half-Length Fix)
# ---------------------------------------------------------
cum_columns = ["goals", "assists", "shots", "passes", "dribbles", "tackles", 
               "possession_won", "possession_lost", "fouls_committed", "offsides", 
               "distance_covered", "distance_sprinted"]

h_base = 10.0

for col in cum_columns:
    # We overwrite the raw stat to represent a standard 10-minute half game.
    # This ensures 5-min half games don't corrupt our dataset averages later.
    st_df[col] = st_df[col] * (h_base / st_df["half_length"])


# ---------------------------------------------------------
# Step 2: Set Bayesian Smoothing Parameters
# ---------------------------------------------------------
dummy_minutes = 30.0  # Shrinkage anchor (requires ~a half of football to break away from mean)


# ---------------------------------------------------------
# Step 3: Apply Smoothed Per-90 Scaling
# ---------------------------------------------------------
# A. Rare Stats (Assume 0 for dummy minutes)
rare_cols = ["goals", "assists", "shots"]
for col in rare_cols:
    st_df[f"{col}_p90"] = (st_df[col] / (st_df["minutes_played"] + dummy_minutes)) * 90.0


# B. Volume Stats (Assume dataset average for dummy minutes)
volume_cols = ["passes", "dribbles", "tackles", "possession_won", "possession_lost", 
               "fouls_committed", "offsides", "distance_covered", "distance_sprinted"]

for col in volume_cols:
    # 1. Calculate the true, half-length-adjusted dataset average Per 90
    league_avg_p90 = (st_df[col].sum() / st_df["minutes_played"].sum()) * 90.0
    
    # 2. Calculate the dummy stat allocation for 45 minutes
    dummy_stat = league_avg_p90 * (dummy_minutes / 90.0)
    
    # 3. Apply the final smoothed formula
    st_df[f"{col}_p90"] = ((st_df[col] + dummy_stat) / (st_df["minutes_played"] + dummy_minutes)) * 90.0

In [242]:
st_df["xT_bonus_p90"] = 0.25 * (st_df["distance_sprinted_p90"] / st_df["distance_covered_p90"]) * np.log(st_df["pass_accuracy"] * st_df["passes_p90"] + 1)

In [243]:
# Match supremacy scalar
gamma = 0.2
delta_xg = (st_df['team_xg'] + 1) / (st_df['opponent_xg'] + 1)
match_supremacy_scalar = gamma * np.log(delta_xg)

In [244]:
# Now we import weights from root/workshop/st_weights.json
# Which is in the format {"goals_p90": 0.3, "assists_p90": 0.2, ...}
col_names = ["goals_p90", "assists_p90", "shots_p90", "shot_accuracy", "passes_p90", "pass_accuracy",
              "dribbles_p90", "dribble_success_rate", "tackles_p90", "tackle_success_rate", "offsides_p90",
              "fouls_committed_p90", "possession_won_p90", "possession_lost_p90", "distance_covered_p90", "distance_sprinted_p90", "xT_bonus_p90"]
weights_path = project_root / "workshop" / "st_weights.json"
with open(weights_path, "r") as f:
    weights_dict = json.load(f)
final_weights = np.array([weights_dict[col] for col in col_names])

In [245]:
# Now we import the means and stds from root/workshop/st_means_stds.json
# Which is in the format {"goals_p90": {"mean": 0.1, "std": 0.2}, "assists_p90": {"mean": 0.05, "std": 0.1}, ...}
means_stds_path = project_root / "workshop" / "st_means_stds.json"
with open(means_stds_path, "r") as f:
    means_stds_dict = json.load(f)

In [246]:
# Compute final z-scores
negative_stats = ["fouls_committed_p90", "possession_lost_p90", "offsides_p90"]
for col in col_names:
    mean = means_stds_dict[col]["mean"]
    std = means_stds_dict[col]["std"]
    if std == 0:
        st_df[f"{col}_z"] = 0
    elif col in negative_stats:
        st_df[f"{col}_z"] = (mean - st_df[col]) / std
    else:
        st_df[f"{col}_z"] = (st_df[col] - mean) / std

In [247]:
# 1. The Ghosting Floor (Limit the bleeding, but let them be punished)
# -1.0 means they will lose a full rating point for doing absolutely nothing, 
# but it stops them from getting a 2.0/10 rating just because they didn't shoot.
ghosting_stats = ['goals_p90_z', 'assists_p90_z', 'shots_p90_z']
for col in ghosting_stats:
    st_df[col] = np.clip(st_df[col], a_min=-1.0, a_max=None)

# 2. Striker Volatility Floor
# Strikers operate in high-traffic areas. 0% accuracy shouldn't ruin a game if they ran 11km.
efficiency_stats = ['shot_accuracy_z', 'pass_accuracy_z', 'dribble_success_rate_z']
for col in efficiency_stats:
    st_df[col] = np.clip(st_df[col], a_min=-0.75, a_max=None)

# 3. Cap the Maximum Z-Score for Goals/Assists
# This protects the 10.0 scale because the +0.8 and +0.6 manual bonuses are applied later.
# st_df["goals_p90_z"] = np.clip(st_df["goals_p90_z"], a_min=None, a_max=1.5)
# st_df["assists_p90_z"] = np.clip(st_df["assists_p90_z"], a_min=None, a_max=1.5)

# 4. Cap the Offsides/Discipline Penalty
# If a striker gets caught offside 5 times, it should hurt (e.g., dropping them an 8.5 to a 7.9),
# but it shouldn't mathematically erase a 2-goal performance. 
st_df["offsides_p90_z"] = np.clip(st_df["offsides_p90_z"], a_min=-1.5, a_max=None)

In [248]:
z_score_cols = [f"{col}_z" for col in col_names]
z_matrix = st_df[z_score_cols].values
st_df['raw_score'] = np.dot(z_matrix, final_weights)

In [249]:
# Match Winners
st_df['raw_score'] += np.where(st_df['goals_p90_z'] > 0, st_df['goals_p90_z'] * 1.2, 0)
st_df['raw_score'] += np.where(st_df['assists_p90_z'] > 0, st_df['assists_p90_z'] * 0.6, 0)

# The Target Man / Complete Forward Bonus
# If they drop deep and put on a passing/carrying masterclass, give them a secondary boost
st_df['raw_score'] += np.where(st_df['passes_p90_z'] > 2, (st_df['passes_p90_z'] - 2) * 0.2, 0)
st_df['raw_score'] += np.where(st_df['dribbles_p90_z'] > 2, (st_df['dribbles_p90_z'] - 2) * 0.2, 0)

In [250]:
k = 0.85
s_0 = np.log(2/3) / k
# Create a new column, applying the sigmoid transformation to the raw score
st_df['raw_rating'] = 10 * (1 / (1 + np.exp(-k * (st_df['raw_score'] - s_0))))

In [251]:
st_df['final_rating'] = st_df['raw_rating'] - match_supremacy_scalar

In [252]:
st_df['final_rating'] = st_df['final_rating'].apply(lambda x: max(0, min(10, x)))

In [255]:
random_row = st_df.sample(n=1).iloc[0]
for col in st_df.columns:
    print(f"{col}: {random_row[col]:.2f},")

match_id: 3.00,
player_id: 32.00,
goals: 0.00,
assists: 1.00,
shots: 6.00,
shot_accuracy: 50.00,
passes: 9.00,
pass_accuracy: 100.00,
dribbles: 11.00,
dribble_success_rate: 91.00,
tackles: 0.00,
tackle_success_rate: 20.91,
offsides: 0.00,
fouls_committed: 0.00,
possession_won: 0.00,
possession_lost: 6.00,
minutes_played: 55.00,
distance_covered: 6.10,
distance_sprinted: 1.40,
half_length: 10.00,
team_possession: 62.00,
team_xg: 10.10,
opponent_xg: 0.60,
goals_p90: 0.00,
assists_p90: 1.06,
shots_p90: 6.35,
passes_p90: 14.22,
dribbles_p90: 16.33,
tackles_p90: 0.56,
possession_won_p90: 0.31,
possession_lost_p90: 8.25,
fouls_committed_p90: 0.02,
offsides_p90: 0.11,
distance_covered_p90: 10.11,
distance_sprinted_p90: 2.33,
xT_bonus_p90: 0.42,
goals_p90_z: -0.64,
assists_p90_z: 1.96,
shots_p90_z: 2.23,
shot_accuracy_z: -0.52,
passes_p90_z: 0.45,
pass_accuracy_z: 1.51,
dribbles_p90_z: 0.92,
dribble_success_rate_z: 0.71,
tackles_p90_z: -1.03,
tackle_success_rate_z: 0.00,
offsides_p90_z: 0.46,
